# Spark Cluster Configuration — All Parameters

This notebook displays **all available** Spark parameters including those
not explicitly set (showing their default or runtime values).

> ⚠️ Spark has thousands of parameters. This notebook groups them by prefix
> and provides filtering and search.

**Notebook 1** (`01_active_configuration`) shows only the explicitly configured
parameters with explanations. Use this notebook for exploration and debugging.


In [1]:
from pyspark.sql import SparkSession
import pandas as pd

MASTER = 'spark://spark-master:7077'
spark = (SparkSession.builder
    .appName('cluster-config-all')
    .master(MASTER)
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 22:00:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2


In [2]:
# Load ALL parameters from the running SparkSession
all_conf = spark.sparkContext.getConf().getAll()
print(f"Total parameters loaded: {len(all_conf)}")

df_all = pd.DataFrame(all_conf, columns=['Parameter', 'Value'])
df_all = df_all.sort_values('Parameter').reset_index(drop=True)
df_all['Prefix'] = df_all['Parameter'].str.split('.').str[:3].str.join('.')

print("\nTop 20 most common prefixes:")
print(df_all['Prefix'].value_counts().head(20).to_string())


Total parameters loaded: 37

Top 20 most common prefixes:
Prefix
spark.sql.catalog                  4
spark.hadoop.fs                    2
spark.sql.adaptive                 2
spark.app.id                       1
spark.driver.extraJavaOptions      1
spark.driver.host                  1
spark.app.startTime                1
spark.app.name                     1
spark.driver.port                  1
spark.eventLog.dir                 1
spark.executor.cores               1
spark.eventLog.enabled             1
spark.executor.extraJavaOptions    1
spark.executor.id                  1
spark.driver.memory                1
spark.app.submitTime               1
spark.history.fs                   1
spark.executor.memory              1
spark.rdd.compress                 1
spark.history.ui                   1


In [3]:
# ── Filter by prefix ────────────────────────────────────────────────────────
# Change prefix_filter to explore a specific category:
#   'spark.sql'            — all SQL parameters
#   'spark.executor'       — executor settings
#   'spark.driver'         — driver settings
#   'spark.sql.adaptive'   — AQE only
#   'spark.sql.catalog'    — catalog settings
#   'spark.shuffle'        — shuffle settings
#   ''                     — ALL parameters

prefix_filter = 'spark.sql'   # ← change here

if prefix_filter:
    filtered = df_all[df_all['Parameter'].str.startswith(prefix_filter)]
else:
    filtered = df_all

print(f"Parameters with prefix '{prefix_filter}': {len(filtered)}")
print()

try:
    from IPython.display import display, HTML
    display(HTML(filtered[['Parameter','Value']].to_html(index=False, border=1,
        justify='left').replace('<th>', '<th style="text-align:left;padding:4px 8px;background:#2c3e50;color:white">').replace('<td>', '<td style="padding:4px 8px;border-bottom:1px solid #ddd;font-size:13px">')))
except ImportError:
    pd.set_option('display.max_colwidth', 80)
    print(filtered[['Parameter','Value']].to_string(index=False))


Parameters with prefix 'spark.sql': 11



Parameter,Value
spark.sql.adaptive.coalescePartitions.enabled,true
spark.sql.adaptive.enabled,true
spark.sql.ansi.enabled,false
spark.sql.artifact.isolation.enabled,false
spark.sql.catalog.local,org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.local.type,hadoop
spark.sql.catalog.local.warehouse,/workspace/data/warehouse/iceberg
spark.sql.catalog.spark_catalog,org.apache.spark.sql.delta.catalog.DeltaCatalog
spark.sql.execution.arrow.pyspark.enabled,true
spark.sql.extensions,"org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,io.delta.sql.DeltaSparkSessionExtension"


In [4]:
# ── Explicitly set vs default ────────────────────────────────────────────────
explicitly_set = {
    'spark.master', 'spark.executor.memory', 'spark.driver.memory',
    'spark.executor.cores', 'spark.sql.shuffle.partitions',
    'spark.serializer', 'spark.sql.adaptive.enabled',
    'spark.sql.adaptive.coalescePartitions.enabled', 'spark.eventLog.enabled',
    'spark.eventLog.dir', 'spark.history.fs.logDirectory',
    'spark.history.ui.port', 'spark.sql.extensions',
    'spark.sql.catalog.local', 'spark.sql.catalog.local.type',
    'spark.sql.catalog.local.warehouse', 'spark.sql.catalog.spark_catalog',
    'spark.sql.execution.arrow.pyspark.enabled', 'spark.sql.ansi.enabled',
    'spark.driver.extraJavaOptions', 'spark.executor.extraJavaOptions',
    'spark.shuffle.sort.bypassMergeThreshold',
}

df_all['Explicitly Set'] = df_all['Parameter'].isin(explicitly_set).map(
    {True: '✅ Yes', False: '— Default'})

print("Explicitly set vs default:")
print(df_all['Explicitly Set'].value_counts().to_string())
print()

set_params = df_all[df_all['Explicitly Set']=='✅ Yes'][['Parameter','Value','Explicitly Set']]
try:
    from IPython.display import display, HTML
    display(HTML(set_params.to_html(index=False, border=1,
        justify='left').replace('<th>', '<th style="text-align:left;padding:4px 8px;background:#27ae60;color:white">').replace('<td>', '<td style="padding:4px 8px;border-bottom:1px solid #ddd">')))
except ImportError:
    print(set_params.to_string(index=False))


Explicitly set vs default:
Explicitly Set
✅ Yes        22
— Default    15



Parameter,Value,Explicitly Set
spark.driver.extraJavaOptions,-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false -Dio.netty.tryReflectionSetAccessible=true -Dlog4j2.logger.gluten_fallback.name=org.apache.gluten.extension.columnar.transition.GlutenFallbackReporter -Dlog4j2.logger.gluten_fallback.level=ERROR,✅ Yes
spark.driver.memory,1g,✅ Yes
spark.eventLog.dir,/opt/spark/logs,✅ Yes
spark.eventLog.enabled,true,✅ Yes
spark.executor.cores,2,✅ Yes
spark.executor.extraJavaOptions,-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false -Dio.netty.tryReflectionSetAccessible=true -Dlog4j2.logger.gluten_fallback.name=org.apache.gluten.extension.columnar.transition.GlutenFallbackReporter -Dlog4j2.logger.gluten_fallback.level=ERROR,✅ Yes
spark.executor.memory,2g,✅ Yes
spark.history.fs.logDirectory,/opt/spark/logs,✅ Yes
spark.history.ui.port,18080,✅ Yes
spark.master,spark://spark-master:7077,✅ Yes


In [5]:
# ── Search parameters by keyword ─────────────────────────────────────────────
# Change search_term to find relevant parameters:
search_term = 'memory'   # ← change here

results = df_all[
    df_all['Parameter'].str.contains(search_term, case=False) |
    df_all['Value'].str.contains(search_term, case=False)
]
print(f"Parameters containing '{search_term}': {len(results)}")
print()
try:
    from IPython.display import display, HTML
    display(HTML(results[['Parameter','Value']].to_html(index=False, border=1,
        justify='left').replace('<th>', '<th style="text-align:left;padding:4px 8px;background:#8e44ad;color:white">').replace('<td>', '<td style="padding:4px 8px;border-bottom:1px solid #ddd">')))
except ImportError:
    print(results[['Parameter','Value']].to_string(index=False))


Parameters containing 'memory': 2



Parameter,Value
spark.driver.memory,1g
spark.executor.memory,2g


In [6]:
# ── Export to CSV ────────────────────────────────────────────────────────────
import pathlib
out = '/workspace/data/spark_all_config.csv'
df_all[['Parameter','Value','Explicitly Set']].to_csv(out, index=False)
print(f"Saved to: {out}")
print(f"Total parameters: {len(df_all)}")


Saved to: /workspace/data/spark_all_config.csv
Total parameters: 37
